# Cejudo 2025 파이프라인 — 통합 코드

```
Step 1  데이터 수집 (CASAS 데이터셋)
Step 2  데이터 전처리  (hex 디코딩 → 16개 특징 → MinMaxScaler)
         └ Step 2-4: 데이터 분할 (70% 학습 / 30% 검증) ← Notebook B 채택
Step 3  다음날 예측 (Attention RNN)
         └ Dropout + EarlyStopping + 학습/검증 loss 분리  ← Notebook B 채택
         └ hex 원본값 직접 파싱 (bath_count 등 정확하게)  ← Notebook B 채택
Step 4  2단계 이상 감지
         └ 1단계: z-score  (p=0.925, z>1.44)
         └ 2단계: 박스플롯 검증  ← 두 노트북 공통
Step 5  결과 해석 (F1, Precision, Recall + 시각화)
         └ Unseen 테스트 데이터(SET7, SET8) 평가  ← Notebook A 채택
```
---
**두 노트북 비교 요약**

| 항목 | Notebook A | Notebook B | 채택 |
|------|-----------|-----------|------|
| 데이터 분할 | ❌ 없음 (학습=평소, 테스트=응급/사망) | ✅ 70/30 시계열 분할 | **B** |
| hex 파싱 | 집계값 사용 (indoor_active_ratio 고정값) | 원본 hex 직접 파싱 | **B** |
| 모델 dropout | ❌ 없음 | ✅ 0.2 | **B** |
| Early stopping | ❌ 없음 | ✅ patience=10 | **B** |
| 검증 loop | ❌ 없음 | ✅ val_loss 모니터링 | **B** |
| Unseen 테스트 (SET7/8) | ✅ 있음 | ❌ 없음 | **A** |
| 특징별 MAE 분해 | ❌ 없음 | ✅ 어떤 특징이 틀리는지 확인 | **B** |
| 최종 평가 (F1 등) | ✅ 있음 | ✅ 있음 | **공통** |


In [ ]:
# ════════════════════════════════════════════════════════════
# 라이브러리 설치 및 임포트
# ════════════════════════════════════════════════════════════
!pip install torch pandas numpy scikit-learn matplotlib seaborn joblib -q

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

print('✅ 라이브러리 임포트 완료')
print(f'PyTorch 버전: {torch.__version__}')
print(f'CUDA 사용 가능: {torch.cuda.is_available()}')

In [ ]:
# ════════════════════════════════════════════════════════════
# 경로 설정
# ════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/ADL데이터'  # 필요 시 경로 수정
os.makedirs(BASE, exist_ok=True)
print(f'✅ 드라이브 마운트 완료: {BASE}')

---
## Step 2 — 데이터 전처리
### Step 2-1: hex 디코딩 함수

In [ ]:
# ════════════════════════════════════════════════════════════
# Step 2-1. hex 디코딩 함수
# Notebook B 방식 채택: 원본 hex를 직접 파싱 → 정확한 특징 추출
# ════════════════════════════════════════════════════════════

def decode_place(raw: str) -> list:
    """place_code_1_list → 1440개 문자열 리스트 ['00','14','FE'...]"""
    raw = str(raw)
    return [raw[i:i+2] for i in range(0, 2880, 2)]


def decode_aix(raw: str) -> np.ndarray:
    """AIX_1_list → 1440개 정수 배열 (0~1000)"""
    return np.array([int(str(raw)[i:i+4], 16) for i in range(0, 5760, 4)])


def decode_sleep(raw: str) -> np.ndarray:
    """sleep_depth_1_list → 1440개 정수 배열 (0~4) / 0x 접두어 자동 제거"""
    s = str(raw)
    if s.startswith('0x'):
        s = s[2:]
    s = s[:2880]
    return np.array([int(s[i:i+2], 16) for i in range(0, 2880, 2)])


def decode_outgoing(raw: str) -> list:
    """outgoing_1_list → 1440개 문자열 리스트 ['FF','FE'...]"""
    raw = str(raw)
    return [raw[i:i+2] for i in range(0, 2880, 2)]


print('✅ Step 2-1. hex 디코딩 함수 정의 완료')

### Step 2-2: 16개 특징 추출

In [ ]:
# ════════════════════════════════════════════════════════════
# Step 2-2. 16개 특징 추출
# Notebook B 방식 채택: 원본 hex 직접 파싱으로 indoor_active_ratio,
#                        bath_count 등 정확하게 계산
# ════════════════════════════════════════════════════════════

FEATURE_NAMES = [
    # 수면 (4개)
    'total_sleep_period',    # 수면 총 시간(분)
    'deep_sleep_ratio',      # 깊은수면(04) 비율
    'sleep_fragmentation',   # 수면 중 각성 횟수
    'sleep_start_hour',      # 취침 시각 (시간 단위)
    # 활동 (4개)
    'aix_mean',              # 하루 평균 AIX
    'aix_active_ratio',      # AIX>0 분 / 1440
    'indoor_active_ratio',   # place=14 분 비율
    'aix_night_ratio',       # 야간(22~06시) AIX 비율
    # 외출 (4개)
    'outgoing_count',        # 외출 횟수
    'outgoing_total_min',    # 외출 총 시간(분)
    'outgoing_late_night',   # 야간 외출 횟수
    'outgoing_avg_duration', # 평균 외출 시간(분)
    # 화장실 (2개)
    'bath_count',            # 화장실 횟수
    'bath_in_sleep',         # 수면 중 화장실 횟수
    # 환경 (2개)
    'temp_range',            # 일교차 (max - min)
    'illu_active_hours',     # 조도>0 시간 수
]


def extract_features(row: pd.Series) -> dict:
    """DataFrame 1행 → 16개 특징 딕셔너리 (Notebook B 방식: hex 직접 파싱)"""

    # hex 디코딩
    place_m = decode_place(row.get('place_code_1_list', 'FE' * 1440))
    aix_m   = decode_aix(str(row.get('AIX_1_list', '0000' * 1440)))
    sleep_m = decode_sleep(str(row.get('sleep_depth_1_list', '00' * 1440)))
    out_m   = decode_outgoing(str(row.get('outgoing_1_list', 'FE' * 1440)))

    temp_arr = [float(row.get(f'temp_{h:02d}', 0)) for h in range(24)]
    illu_arr = [float(row.get(f'illu_{h:02d}', 0)) for h in range(24)]

    feat = {}

    # ── 수면 (4개) ───────────────────────────────────────────
    slp_idx = np.where(sleep_m > 0)[0]
    feat['total_sleep_period']  = len(slp_idx)
    feat['deep_sleep_ratio']    = int(np.sum(sleep_m == 4)) / 1440

    frag = 0
    in_sleep = False
    for s in sleep_m:
        if s > 0 and not in_sleep:  in_sleep = True
        elif s == 0 and in_sleep:   in_sleep = False; frag += 1
    feat['sleep_fragmentation'] = frag
    feat['sleep_start_hour']    = float(slp_idx[0]) / 60 if len(slp_idx) > 0 else 0.0

    # ── 활동 (4개) ───────────────────────────────────────────
    feat['aix_mean']            = float(aix_m.mean())
    feat['aix_active_ratio']    = float((aix_m > 0).sum()) / 1440
    feat['indoor_active_ratio'] = sum(1 for p in place_m if p == '14') / 1440  # 직접 파싱
    night_aix = np.concatenate([aix_m[22*60:], aix_m[:6*60]])
    total_aix = aix_m.sum()
    feat['aix_night_ratio']     = float(night_aix.sum()) / float(total_aix) if total_aix > 0 else 0.0

    # ── 외출 (4개) ───────────────────────────────────────────
    out_sessions = []
    in_out = False; os_ = 0
    for i, o in enumerate(out_m):
        if o == 'FF' and not in_out:  in_out = True;  os_ = i
        elif o != 'FF' and in_out:    in_out = False;  out_sessions.append((os_, i - os_))
    if in_out:
        out_sessions.append((os_, 1440 - os_))

    night_out = [s for s, d in out_sessions if s >= 22*60 or s < 6*60]
    feat['outgoing_count']        = len(out_sessions)
    feat['outgoing_total_min']    = sum(d for _, d in out_sessions)
    feat['outgoing_late_night']   = len(night_out)
    feat['outgoing_avg_duration'] = (
        feat['outgoing_total_min'] / feat['outgoing_count']
        if feat['outgoing_count'] > 0 else 0.0
    )

    # ── 화장실 (2개) ─────────────────────────────────────────
    # place='14' 연속 구간 2~30분 → 화장실 방문으로 간주
    bath_segs = []
    in_b = False; bs = 0
    for i, p in enumerate(place_m):
        if p == '14' and not in_b:  in_b = True;  bs = i
        elif p != '14' and in_b:
            in_b = False
            dur = i - bs
            if 2 <= dur <= 30:
                bath_segs.append((bs, dur))
    slp_set = set(slp_idx.tolist())
    feat['bath_count']    = len(bath_segs)
    feat['bath_in_sleep'] = sum(1 for s, _ in bath_segs if s in slp_set)

    # ── 환경 (2개) ───────────────────────────────────────────
    feat['temp_range']        = max(temp_arr) - min(temp_arr) if temp_arr else 0.0
    feat['illu_active_hours'] = sum(1 for v in illu_arr if v > 0)

    return feat


def build_feature_df(df_raw: pd.DataFrame, label: int) -> pd.DataFrame:
    """전체 DataFrame → 특징 DataFrame"""
    rows = []
    for _, row in df_raw.iterrows():
        feat = extract_features(row)
        feat['patient_id'] = row.get('care_recipient_id', row.get('patient_id', 'unknown'))
        feat['date']       = row.get('lifeog_date', row.get('date', None))
        feat['label']      = label
        rows.append(feat)
    df_feat = pd.DataFrame(rows)
    return df_feat[['patient_id', 'date'] + FEATURE_NAMES + ['label']]


print('✅ Step 2-2. 16개 특징 추출 함수 정의 완료')

### Step 2-3: 데이터 로드 및 정규화

In [ ]:
# ════════════════════════════════════════════════════════════
# Step 2-3. 데이터 로드 (SET1~SET6) + 특징 추출 + 정규화
# ════════════════════════════════════════════════════════════

print('[ SET1~SET6 로드 중 ]')
normal_list, emrg_list, death_list = [], [], []

for s in range(1, 7):
    df_n = pd.read_csv(f'{BASE}/SET{s}_평소ADL_5명_60일.csv', encoding='utf-8-sig')
    df_e = pd.read_csv(f'{BASE}/SET{s}_응급ADL_5명_60일.csv', encoding='utf-8-sig')
    df_d = pd.read_csv(f'{BASE}/SET{s}_사망전ADL_5명_60일.csv', encoding='utf-8-sig')

    normal_list.append(build_feature_df(df_n, label=0))
    emrg_list.append(build_feature_df(df_e,  label=1))
    death_list.append(build_feature_df(df_d,  label=2))
    print(f'  SET{s} 완료')

df_normal = pd.concat(normal_list).reset_index(drop=True)
df_emrg   = pd.concat(emrg_list).reset_index(drop=True)
df_death  = pd.concat(death_list).reset_index(drop=True)

print(f'\n  평소: {df_normal.shape}  응급: {df_emrg.shape}  사망: {df_death.shape}')

# ── 정규화: 평소 ADL만으로 fit ───────────────────────────────
print('\n[ MinMaxScaler 정규화 ]')
scaler = MinMaxScaler()

X_normal_scaled = scaler.fit_transform(df_normal[FEATURE_NAMES].values)  # 평소 기준 fit
X_emrg_scaled   = scaler.transform(df_emrg[FEATURE_NAMES].values)
X_death_scaled  = scaler.transform(df_death[FEATURE_NAMES].values)

df_normal_scaled = df_normal.copy(); df_normal_scaled[FEATURE_NAMES] = X_normal_scaled
df_emrg_scaled   = df_emrg.copy();   df_emrg_scaled[FEATURE_NAMES]   = X_emrg_scaled
df_death_scaled  = df_death.copy();  df_death_scaled[FEATURE_NAMES]  = X_death_scaled

print(f'  평소 범위: {X_normal_scaled.min():.3f} ~ {X_normal_scaled.max():.3f}  ✅')
print(f'  응급 최대: {X_emrg_scaled.max():.3f}')
print(f'  사망 최대: {X_death_scaled.max():.3f}')

# scaler 저장
joblib.dump(scaler,       f'{BASE}/scaler.pkl')
joblib.dump(FEATURE_NAMES, f'{BASE}/feature_names.pkl')
print(f'\n✅ Step 2-3. 정규화 완료  →  scaler 저장: {BASE}/scaler.pkl')

### Step 2-4: 데이터 분할 (70/30) + 슬라이딩 윈도우

In [ ]:
# ════════════════════════════════════════════════════════════
# Step 2-4. 데이터 분할 + 슬라이딩 윈도우
#
# Notebook B 방식 채택:
#   - 평소 ADL을 환자별로 시간 순서 유지하며 70/30 분할
#   - 학습: 앞 70% 날짜, 검증: 뒤 30% 날짜
#   - random shuffle 금지 (시계열 데이터)
#
# Notebook A 문제점: 분할 없이 SET1~6(평소)=학습, SET7~8(응급/사망)=테스트
#   → 학습/검증 독립성이 보장되지 않아 과도한 성능 수치가 나올 수 있음
# ════════════════════════════════════════════════════════════

# 윈도우 크기: 논문 권장 15일, 실험적으로 5일도 가능
WINDOW_SIZE = 15  # 논문 권장값
TRAIN_RATIO = 0.7


def split_by_patient(df: pd.DataFrame, train_ratio: float = 0.7) -> tuple:
    """
    환자별 시간 순서 유지 70/30 분할
    60일 × 30명 → 학습 42일 × 30명 / 검증 18일 × 30명
    """
    train_list, test_list = [], []
    for pid in df['patient_id'].unique():
        df_p = df[df['patient_id'] == pid].sort_values('date').reset_index(drop=True)
        cut  = int(len(df_p) * train_ratio)
        train_list.append(df_p.iloc[:cut])
        test_list.append(df_p.iloc[cut:])
    return (pd.concat(train_list).reset_index(drop=True),
            pd.concat(test_list).reset_index(drop=True))


def create_windows(df: pd.DataFrame, window_size: int = WINDOW_SIZE) -> tuple:
    """
    환자별 슬라이딩 윈도우 생성
    입력: 연속 window_size일치 특징 벡터
    출력: 다음날 특징 벡터
    환자 경계에서 절대 이어 붙이지 않음
    """
    X_list, y_list, meta_list = [], [], []
    for pid in df['patient_id'].unique():
        df_p   = df[df['patient_id'] == pid].sort_values('date').reset_index(drop=True)
        n      = len(df_p)
        if n <= window_size:
            print(f'  경고: 환자 {pid} 데이터({n}일) ≤ 윈도우({window_size}일) → 스킵')
            continue
        values = df_p[FEATURE_NAMES].values
        dates  = df_p['date'].values
        for i in range(n - window_size):
            X_list.append(values[i : i + window_size])
            y_list.append(values[i + window_size])
            meta_list.append({
                'patient_id':  pid,
                'window_start': dates[i],
                'target_date':  dates[i + window_size],
                'label':        df_p['label'].iloc[i + window_size],
            })
    return np.array(X_list), np.array(y_list), meta_list


# ── 분할 실행 ────────────────────────────────────────────────
df_train, df_val = split_by_patient(df_normal_scaled, TRAIN_RATIO)

print('[ 데이터 분할 결과 ]')
print(f'  전체 평소 ADL : {len(df_normal_scaled)}일')
print(f'  학습용        : {len(df_train)}일 (앞 {TRAIN_RATIO*100:.0f}%)')
print(f'  검증용        : {len(df_val)}일  (뒤 {(1-TRAIN_RATIO)*100:.0f}%)')

# ── 슬라이딩 윈도우 생성 ────────────────────────────────────
X_train, y_train, meta_train = create_windows(df_train, WINDOW_SIZE)
X_val,   y_val,   meta_val   = create_windows(df_val,   WINDOW_SIZE)

# 응급/사망 이상 감지용 윈도우
X_emrg,  y_emrg,  meta_emrg  = create_windows(df_emrg_scaled,  WINDOW_SIZE)
X_death, y_death, meta_death  = create_windows(df_death_scaled, WINDOW_SIZE)

print(f'\n[ 슬라이딩 윈도우 결과 ]  (윈도우 크기: {WINDOW_SIZE}일)')
print(f'  X_train : {X_train.shape}  →  (샘플수, {WINDOW_SIZE}일, 16특징)')
print(f'  X_val   : {X_val.shape}')
print(f'  X_emrg  : {X_emrg.shape}')
print(f'  X_death : {X_death.shape}')

print(f'\n✅ Step 2-4. 데이터 분할 + 슬라이딩 윈도우 완료')

---
## Step 3 — 다음날 예측 (Attention RNN)

In [ ]:
# ════════════════════════════════════════════════════════════
# Step 3. Attention RNN 모델
#
# Notebook B 방식 채택:
#   - Attention 모듈 분리 (attn weight 반환 → 해석 가능)
#   - LSTM dropout=0.2 (과적합 방지)
#   - train/val loss 모두 모니터링
#   - EarlyStopping patience=10
# ════════════════════════════════════════════════════════════

class ADLDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]


class Attention(nn.Module):
    """
    window_size일치 LSTM 출력 중 어느 날이 중요한지 가중치 계산
    입력: (batch, window_size, hidden)
    출력: (batch, hidden) 가중합
    """
    def __init__(self, hidden_size: int):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1)

    def forward(self, lstm_out):
        score  = self.attn(lstm_out)           # (batch, window, 1)
        weight = torch.softmax(score, dim=1)   # 날별 중요도
        context = (weight * lstm_out).sum(dim=1)  # (batch, hidden)
        return context, weight


class AttentionRNN(nn.Module):
    """
    Input  → (batch, window_size, 16)
    LSTM   → (batch, window_size, 64)  with dropout=0.2
    Attention → (batch, 64)
    Dense(ReLU) → (batch, 32)
    Output → (batch, 16)  다음날 16개 특징 예측
    """
    def __init__(self, input_size: int = 16, hidden_size: int = 64, output_size: int = 16):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            batch_first = True,
            dropout     = 0.2,   # Notebook B 채택: 과적합 방지
        )
        self.attention = Attention(hidden_size)
        self.fc1  = nn.Linear(hidden_size, 32)
        self.relu = nn.ReLU()
        self.fc2  = nn.Linear(32, output_size)

    def forward(self, x):
        lstm_out, _          = self.lstm(x)
        context, attn_weight = self.attention(lstm_out)
        out = self.relu(self.fc1(context))
        out = self.fc2(out)
        return out, attn_weight


print('✅ Step 3. Attention RNN 모델 정의 완료')
model_test = AttentionRNN()
total_params = sum(p.numel() for p in model_test.parameters())
print(f'   총 파라미터 수: {total_params:,}')

In [ ]:
# ════════════════════════════════════════════════════════════
# Step 3. 모델 학습
#
# Notebook B 방식 채택:
#   - val_loss 기반 EarlyStopping (patience=10)
#   - 가장 좋은 val_loss 시점의 모델 저장
#   - train/val loss 동시 모니터링
# ════════════════════════════════════════════════════════════

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_dataset = ADLDataset(X_train, y_train)
val_dataset   = ADLDataset(X_val,   y_val)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False)

model     = AttentionRNN().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS   = 100
PATIENCE = 10

best_val_loss = float('inf')
patience_cnt  = 0
history = {'train': [], 'val': []}

print(f'[ 학습 시작 ]  device={device}  train={len(train_dataset)}  val={len(val_dataset)}')
print(f'  epochs={EPOCHS}, patience={PATIENCE}, batch=16, lr=0.001\n')

for epoch in range(EPOCHS):

    # ── Train ────────────────────────────────────────────────
    model.train()
    train_loss = 0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        pred, _ = model(X_b)
        loss = criterion(pred, y_b)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # ── Validation ───────────────────────────────────────────
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            pred, _ = model(X_b)
            val_loss += criterion(pred, y_b).item()
    val_loss /= len(val_loader)

    history['train'].append(train_loss)
    history['val'].append(val_loss)

    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1:3d}/{EPOCHS} | train={train_loss:.4f} | val={val_loss:.4f}')

    # ── EarlyStopping ────────────────────────────────────────
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_cnt  = 0
        torch.save(model.state_dict(), f'{BASE}/best_model.pt')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'\n  EarlyStopping at epoch {epoch+1}  (best val={best_val_loss:.4f})')
            break

# 최적 모델 로드
model.load_state_dict(torch.load(f'{BASE}/best_model.pt', map_location=device))
model.eval()

# 학습 곡선 시각화
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history['train'], label='Train Loss', color='steelblue')
ax.plot(history['val'],   label='Val Loss',   color='tomato')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('Training / Validation Loss Curve')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{BASE}/loss_curve.png', dpi=150, bbox_inches='tight')
plt.close()

print(f'\n✅ Step 3. 모델 학습 완료  →  저장: {BASE}/best_model.pt')

---
## Step 4 — 2단계 이상 감지
### Step 4-1: MAE 오차 계산

In [ ]:
# ════════════════════════════════════════════════════════════
# Step 4-1. MAE 오차 계산
# Notebook B 방식 채택: 특징별 MAE 분해 → 어떤 특징이 크게 틀리는지 파악
# ════════════════════════════════════════════════════════════

def calc_mae(model, X, y, device, batch_size=64):
    """
    모델 예측 → MAE 계산
    Returns:
        mae_scores : (samples,)    하루당 오차 점수
        y_pred_all : (samples, 16) 예측값 전체
    """
    model.eval()
    mae_list, y_pred_all = [], []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            X_b  = torch.FloatTensor(X[i:i+batch_size]).to(device)
            y_b  = y[i:i+batch_size]
            pred, _ = model(X_b)
            pred = pred.cpu().numpy()
            mae_list.append(np.mean(np.abs(pred - y_b), axis=1))
            y_pred_all.append(pred)
    return np.concatenate(mae_list), np.concatenate(y_pred_all)


print('[ MAE 계산 중... ]')

# 검증 셋 (평소 후반부 30%)
mae_normal, pred_normal = calc_mae(model, X_val,   y_val,   device)

# 이상 감지 대상
mae_emrg,   pred_emrg   = calc_mae(model, X_emrg,  y_emrg,  device)
mae_death,  pred_death   = calc_mae(model, X_death, y_death, device)

print(f'  평소(val) MAE : mean={mae_normal.mean():.4f}, std={mae_normal.std():.4f}')
print(f'  응급     MAE : mean={mae_emrg.mean():.4f},  std={mae_emrg.std():.4f}')
print(f'  사망     MAE : mean={mae_death.mean():.4f},  std={mae_death.std():.4f}')

# ── 특징별 MAE 분해 (Notebook B 채택) ───────────────────────
def feature_mae(y_pred, y_real):
    return np.mean(np.abs(y_pred - y_real), axis=0)  # (16,)

feat_mae_normal = feature_mae(pred_normal, y_val)
feat_mae_emrg   = feature_mae(pred_emrg,   y_emrg)
feat_mae_death  = feature_mae(pred_death,  y_death)

print(f'\n[ 특징별 MAE 비교 — 응급 기준 상위 5개 ]')
print(f'  {"특징명":<25} {"평소":>8} {"응급":>8} {"사망":>8} {"응급/평소":>10}')
print(f'  {"-"*60}')
for i in np.argsort(feat_mae_emrg)[::-1][:5]:
    ratio = feat_mae_emrg[i] / feat_mae_normal[i] if feat_mae_normal[i] > 0 else 0
    print(f'  {FEATURE_NAMES[i]:<25} {feat_mae_normal[i]:>8.4f} '
          f'{feat_mae_emrg[i]:>8.4f} {feat_mae_death[i]:>8.4f} {ratio:>9.1f}x')

print(f'\n✅ Step 4-1. MAE 계산 완료')

### Step 4-2: z-score 이상 판단

In [ ]:
# ════════════════════════════════════════════════════════════
# Step 4-2. z-score 이상 판단
# 논문: p=0.925 → z > 1.44 (상위 7.5%만 이상 의심)
#       허약 노인: p=0.85 → z > 1.04
# ════════════════════════════════════════════════════════════

# 평소 val MAE 기준으로 μ, σ 계산
mu    = mae_normal.mean()
sigma = mae_normal.std()

THRESHOLD_NORMAL = 1.44  # 일반 노인
THRESHOLD_FRAIL  = 1.04  # 허약 노인

print('[ 평소 MAE 분포 (검증 세트 기준) ]')
print(f'  μ={mu:.4f}, σ={sigma:.4f}')


def detect_anomaly(mae_scores, mu, sigma, threshold):
    z_scores   = (mae_scores - mu) / sigma
    is_anomaly = z_scores > threshold
    return z_scores, is_anomaly


z_normal, anomaly_normal = detect_anomaly(mae_normal, mu, sigma, THRESHOLD_NORMAL)
z_emrg,   anomaly_emrg   = detect_anomaly(mae_emrg,   mu, sigma, THRESHOLD_NORMAL)
z_death,  anomaly_death   = detect_anomaly(mae_death,  mu, sigma, THRESHOLD_NORMAL)

print(f'\n[ z-score 이상 판정 (z > {THRESHOLD_NORMAL}) ]')
print(f'  {"라벨":<6} {"샘플":>6} {"이상":>8} {"감지율":>8} {"평균z":>8}')
print(f'  {"-"*42}')
for label, z, a in [("평소", z_normal, anomaly_normal),
                     ("응급", z_emrg, anomaly_emrg),
                     ("사망", z_death, anomaly_death)]:
    print(f'  {label:<6} {len(z):>6} {a.sum():>8} {a.mean()*100:>7.1f}% {z.mean():>8.2f}')

print(f'\n✅ Step 4-2. z-score 이상 판정 완료')

### Step 4-3: 박스플롯 검증 (2단계 확정)

In [ ]:
# ════════════════════════════════════════════════════════════
# Step 4-3. 박스플롯 검증
# z-score 이상 의심 → 어떤 특징이 정상 범위를 벗어났는지 확인
# 두 조건 모두 만족 → 이상 확정
# ════════════════════════════════════════════════════════════

# 정상 범위 계산 (정규화 전 원본값 기준)
X_normal_raw = df_normal[FEATURE_NAMES].values

Q1  = np.percentile(X_normal_raw, 25, axis=0)
Q3  = np.percentile(X_normal_raw, 75, axis=0)
IQR = Q3 - Q1

upper = Q3 + 1.5 * IQR
lower = np.maximum(Q1 - 1.5 * IQR, 0)  # 음수 방지

print('[ 특징별 정상 범위 (Q1-1.5IQR ~ Q3+1.5IQR) ]')
print(f'  {"특징명":<25} {"하한":>8} {"Q1":>8} {"Q3":>8} {"상한":>8}')
print(f'  {"-"*58}')
for i, name in enumerate(FEATURE_NAMES):
    print(f'  {name:<25} {lower[i]:>8.2f} {Q1[i]:>8.2f} {Q3[i]:>8.2f} {upper[i]:>8.2f}')


def check_boxplot(row_values: np.ndarray, lower, upper) -> list:
    """정상 범위 벗어난 특징 목록 반환"""
    outliers = []
    for i, name in enumerate(FEATURE_NAMES):
        val = row_values[i]
        if val > upper[i]:
            outliers.append({'feature': name, 'value': val,
                             'bound': upper[i], 'direction': '상한 초과'})
        elif val < lower[i]:
            outliers.append({'feature': name, 'value': val,
                             'bound': lower[i], 'direction': '하한 미달'})
    return outliers


def run_two_stage_detection(df_feat, mae_scores, z_scores,
                             meta_list, label_name, lower, upper):
    """
    2단계 이상 감지 실행
    1단계: z > THRESHOLD_NORMAL → 이상 의심
    2단계: 박스플롯 이탈 ≥ 1개  → 이상 확정
    """
    results = []
    X_raw   = df_feat[FEATURE_NAMES].values

    for i in range(len(mae_scores)):
        is_suspect   = z_scores[i] > THRESHOLD_NORMAL
        outliers     = check_boxplot(X_raw[i], lower, upper)
        is_confirmed = is_suspect and len(outliers) > 0

        meta = meta_list[i] if i < len(meta_list) else {}
        results.append({
            'patient_id':       meta.get('patient_id', '?'),
            'date':             meta.get('target_date', '?'),
            'label':            label_name,
            'mae':              round(mae_scores[i], 4),
            'z_score':          round(z_scores[i], 2),
            'is_suspect':       is_suspect,
            'is_confirmed':     is_confirmed,
            'outlier_count':    len(outliers),
            'outlier_features': ', '.join([o['feature'] for o in outliers]),
        })
    return pd.DataFrame(results)


# ── 이상 감지 실행 ───────────────────────────────────────────
# 평소는 검증 세트(df_val)를 사용 (학습에 사용하지 않은 데이터)
df_res_normal = run_two_stage_detection(
    df_val.iloc[:len(mae_normal)],  mae_normal, z_normal, meta_val,   '평소', lower, upper)
df_res_emrg   = run_two_stage_detection(
    df_emrg.iloc[:len(mae_emrg)],   mae_emrg,   z_emrg,   meta_emrg,  '응급', lower, upper)
df_res_death  = run_two_stage_detection(
    df_death.iloc[:len(mae_death)], mae_death,  z_death,  meta_death, '사망', lower, upper)

df_all = pd.concat([df_res_normal, df_res_emrg, df_res_death], ignore_index=True)

print(f'\n[ 2단계 이상 확정 결과 ]')
print(f'  {"라벨":<6} {"샘플":>6} {"1단계의심":>10} {"2단계확정":>10} {"확정율":>8}')
print(f'  {"-"*46}')
for lbl in ['평소', '응급', '사망']:
    df_l = df_all[df_all['label'] == lbl]
    s    = df_l['is_suspect'].sum()
    c    = df_l['is_confirmed'].sum()
    print(f'  {lbl:<6} {len(df_l):>6} {s:>10} {c:>10} {c/len(df_l)*100:>7.1f}%')

# ── 이상 확정 예시 출력 ──────────────────────────────────────
print(f'\n[ 이상 확정 예시 (응급 상위 3건) ]')
for _, row in df_res_emrg[df_res_emrg['is_confirmed']].head(3).iterrows():
    print(f"  환자 {row['patient_id']} / {row['date']}  MAE={row['mae']:.4f}  z={row['z_score']:.2f}")
    print(f"    이탈 특징: {row['outlier_features']}")

# CSV 저장
df_all.to_csv(f'{BASE}/이상_감지_결과.csv', index=False, encoding='utf-8-sig')
print(f'\n✅ Step 4-3. 박스플롯 검증 완료  →  결과 저장: {BASE}/이상_감지_결과.csv')

---
## Step 5 — 결과 해석 및 최종 평가

In [ ]:
# ════════════════════════════════════════════════════════════
# Step 5-1. 내부 성능 평가 (SET1~6 검증 세트 기준)
# Precision / Recall / F1 + 혼동 행렬
# ════════════════════════════════════════════════════════════

y_true_normal = np.zeros(len(df_res_normal), dtype=int)
y_true_emrg   = np.ones(len(df_res_emrg),   dtype=int)
y_true_death  = np.ones(len(df_res_death),  dtype=int)

y_pred_normal = df_res_normal['is_confirmed'].astype(int).values
y_pred_emrg   = df_res_emrg['is_confirmed'].astype(int).values
y_pred_death  = df_res_death['is_confirmed'].astype(int).values

y_true = np.concatenate([y_true_normal, y_true_emrg, y_true_death])
y_pred = np.concatenate([y_pred_normal, y_pred_emrg, y_pred_death])

precision = precision_score(y_true, y_pred, zero_division=0)
recall    = recall_score(y_true, y_pred, zero_division=0)
f1        = f1_score(y_true, y_pred, zero_division=0)
cm        = confusion_matrix(y_true, y_pred)

print('[ 내부 성능 평가 (SET1~6 검증 세트) ]')
print(f'  Precision : {precision:.4f}  ({precision*100:.1f}%)')
print(f'  Recall    : {recall:.4f}  ({recall*100:.1f}%)')
print(f'  F1        : {f1:.4f}  ({f1*100:.1f}%)')
print(f'\n  혼동 행렬:')
print(f'            예측:정상  예측:이상')
print(f'  실제:정상  {cm[0,0]:>8}  {cm[0,1]:>8}  ← FP={cm[0,1]}')
print(f'  실제:이상  {cm[1,0]:>8}  {cm[1,1]:>8}  ← FN={cm[1,0]}')

print(f'\n✅ Step 5-1. 내부 성능 평가 완료')

In [ ]:
# ════════════════════════════════════════════════════════════
# Step 5-2. 외부 테스트 (Unseen Data: SET7, SET8)
# Notebook A 방식 채택: 학습에 전혀 사용하지 않은 새 환자 데이터로 평가
# → 학습 데이터 누수 없음, 더 현실적인 성능 지표
# ════════════════════════════════════════════════════════════

print('[ SET7, SET8 Unseen 테스트 데이터 로드 ]')

df_test_normal = pd.concat([
    pd.read_csv(f'{BASE}/SET7_평소ADL_5명_60일.csv', encoding='utf-8-sig'),
    pd.read_csv(f'{BASE}/SET8_평소ADL_5명_60일.csv', encoding='utf-8-sig'),
], ignore_index=True)

df_test_emrg = pd.concat([
    pd.read_csv(f'{BASE}/SET7_응급ADL_5명_60일.csv', encoding='utf-8-sig'),
    pd.read_csv(f'{BASE}/SET8_응급ADL_5명_60일.csv', encoding='utf-8-sig'),
], ignore_index=True)

df_test_death = pd.concat([
    pd.read_csv(f'{BASE}/SET7_사망전ADL_5명_60일.csv', encoding='utf-8-sig'),
    pd.read_csv(f'{BASE}/SET8_사망전ADL_5명_60일.csv', encoding='utf-8-sig'),
], ignore_index=True)

print(f'  평소: {df_test_normal.shape}  응급: {df_test_emrg.shape}  사망: {df_test_death.shape}')

# ── 전처리 (학습 때 저장한 scaler 사용) ─────────────────────
df_feat_tn = build_feature_df(df_test_normal, label=0)
df_feat_te = build_feature_df(df_test_emrg,  label=1)
df_feat_td = build_feature_df(df_test_death, label=2)

# 학습 scaler로 transform (fit 아님!)
for df_f in [df_feat_tn, df_feat_te, df_feat_td]:
    df_f[FEATURE_NAMES] = scaler.transform(df_f[FEATURE_NAMES].values)

# 슬라이딩 윈도우
X_tn, y_tn, meta_tn = create_windows(df_feat_tn, WINDOW_SIZE)
X_te, y_te, meta_te = create_windows(df_feat_te, WINDOW_SIZE)
X_td, y_td, meta_td = create_windows(df_feat_td, WINDOW_SIZE)

print(f'  X_tn(평소): {X_tn.shape}  X_te(응급): {X_te.shape}  X_td(사망): {X_td.shape}')

# ── MAE + z-score ────────────────────────────────────────────
mae_tn, _ = calc_mae(model, X_tn, y_tn, device)
mae_te, _ = calc_mae(model, X_te, y_te, device)
mae_td, _ = calc_mae(model, X_td, y_td, device)

z_tn, anomaly_tn = detect_anomaly(mae_tn, mu, sigma, THRESHOLD_NORMAL)
z_te, anomaly_te = detect_anomaly(mae_te, mu, sigma, THRESHOLD_NORMAL)
z_td, anomaly_td = detect_anomaly(mae_td, mu, sigma, THRESHOLD_NORMAL)

# ── 박스플롯 2단계 ───────────────────────────────────────────
# 원본 특징 DataFrame에서 정규화 전 값 필요 → df_normal 기준 upper/lower 재사용
df_feat_tn_raw = build_feature_df(df_test_normal, label=0)  # 정규화 전 원본
df_feat_te_raw = build_feature_df(df_test_emrg,  label=1)
df_feat_td_raw = build_feature_df(df_test_death, label=2)

df_res_tn = run_two_stage_detection(
    df_feat_tn_raw.iloc[:len(mae_tn)], mae_tn, z_tn, meta_tn, '평소(unseen)', lower, upper)
df_res_te = run_two_stage_detection(
    df_feat_te_raw.iloc[:len(mae_te)], mae_te, z_te, meta_te, '응급(unseen)', lower, upper)
df_res_td = run_two_stage_detection(
    df_feat_td_raw.iloc[:len(mae_td)], mae_td, z_td, meta_td, '사망(unseen)', lower, upper)

# ── Unseen 성능 지표 ─────────────────────────────────────────
yp_tn = df_res_tn['is_confirmed'].astype(int).values
yp_te = df_res_te['is_confirmed'].astype(int).values
yp_td = df_res_td['is_confirmed'].astype(int).values

yt = np.concatenate([np.zeros(len(yp_tn)), np.ones(len(yp_te)), np.ones(len(yp_td))])
yp = np.concatenate([yp_tn, yp_te, yp_td])

p_u = precision_score(yt, yp, zero_division=0)
r_u = recall_score(yt, yp, zero_division=0)
f_u = f1_score(yt, yp, zero_division=0)
cm_u = confusion_matrix(yt, yp)

print(f'\n[ Unseen 성능 평가 (SET7+8) ]')
print(f'  Precision : {p_u:.4f}  ({p_u*100:.1f}%)')
print(f'  Recall    : {r_u:.4f}  ({r_u*100:.1f}%)')
print(f'  F1        : {f_u:.4f}  ({f_u*100:.1f}%)')
print(f'\n  혼동 행렬:')
print(f'            예측:정상  예측:이상')
print(f'  실제:정상  {cm_u[0,0]:>8}  {cm_u[0,1]:>8}  ← FP={cm_u[0,1]}')
print(f'  실제:이상  {cm_u[1,0]:>8}  {cm_u[1,1]:>8}  ← FN={cm_u[1,0]}')

print(f'\n✅ Step 5-2. Unseen 테스트 평가 완료')

In [ ]:
# ════════════════════════════════════════════════════════════
# Step 5-3. 최종 시각화
# ════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. MAE 분포 (내부 검증)
ax = axes[0, 0]
ax.hist(mae_normal, bins=30, alpha=0.7, color='green',  label=f'평소(val) n={len(mae_normal)}')
ax.hist(mae_emrg,   bins=30, alpha=0.7, color='orange', label=f'응급 n={len(mae_emrg)}')
ax.hist(mae_death,  bins=30, alpha=0.7, color='red',    label=f'사망 n={len(mae_death)}')
ax.axvline(mu + THRESHOLD_NORMAL * sigma, color='black',
           linestyle='--', label=f'z>{THRESHOLD_NORMAL}')
ax.set_title('MAE 분포 (내부 검증)'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# 2. MAE 분포 (Unseen)
ax = axes[0, 1]
ax.hist(mae_tn, bins=30, alpha=0.7, color='green',  label=f'평소(unseen)')
ax.hist(mae_te, bins=30, alpha=0.7, color='orange', label=f'응급(unseen)')
ax.hist(mae_td, bins=30, alpha=0.7, color='red',    label=f'사망(unseen)')
ax.axvline(mu + THRESHOLD_NORMAL * sigma, color='black', linestyle='--', label=f'z>{THRESHOLD_NORMAL}')
ax.set_title('MAE 분포 (Unseen SET7+8)'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# 3. 성능 지표 비교
ax = axes[0, 2]
x = np.arange(3)
w = 0.35
ax.bar(x - w/2, [precision, recall, f1],   w, label='내부(SET1~6)', color='steelblue', alpha=0.8)
ax.bar(x + w/2, [p_u, r_u, f_u], w, label='Unseen(SET7+8)', color='tomato', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(['Precision', 'Recall', 'F1'])
ax.set_ylim(0, 1.15); ax.legend(); ax.grid(axis='y', alpha=0.3)
ax.set_title('성능 지표 비교')
for xi, (vi, vu) in enumerate(zip([precision, recall, f1], [p_u, r_u, f_u])):
    ax.text(xi - w/2, vi + 0.02, f'{vi:.3f}', ha='center', fontsize=9, fontweight='bold')
    ax.text(xi + w/2, vu + 0.02, f'{vu:.3f}', ha='center', fontsize=9, fontweight='bold')

# 4. 혼동 행렬 (내부)
ax = axes[1, 0]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['예측:정상', '예측:이상'],
            yticklabels=['실제:정상', '실제:이상'])
ax.set_title('혼동 행렬 (내부 검증)')

# 5. 혼동 행렬 (Unseen)
ax = axes[1, 1]
sns.heatmap(cm_u, annot=True, fmt='d', cmap='Oranges', ax=ax,
            xticklabels=['예측:정상', '예측:이상'],
            yticklabels=['실제:정상', '실제:이상'])
ax.set_title('혼동 행렬 (Unseen SET7+8)')

# 6. 이상 감지율 요약
ax = axes[1, 2]
labels  = ['평소\n(FA↓)', '응급\n(TP↑)', '사망\n(TP↑)']
rates_i = [anomaly_normal.mean()*100, anomaly_emrg.mean()*100, anomaly_death.mean()*100]
rates_u = [anomaly_tn.mean()*100,     anomaly_te.mean()*100,   anomaly_td.mean()*100]
x = np.arange(3)
ax.bar(x - w/2, rates_i, w, label='내부', color=['green','orange','red'], alpha=0.7)
ax.bar(x + w/2, rates_u, w, label='Unseen', color=['green','orange','red'], alpha=0.4, hatch='//')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('z-score 이상 감지율 (%)'); ax.set_ylim(0, 115)
ax.set_title('라벨별 이상 감지율'); ax.legend(); ax.grid(axis='y', alpha=0.3)

plt.suptitle('Cejudo 2025 파이프라인 — 최종 결과', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{BASE}/최종_결과_시각화.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()

print(f'✅ Step 5-3. 시각화 저장: {BASE}/최종_결과_시각화.png')
print(f'\n🎉 전체 파이프라인 완료!')
print(f'   내부:  P={precision:.3f}  R={recall:.3f}  F1={f1:.3f}')
print(f'   Unseen: P={p_u:.3f}  R={r_u:.3f}  F1={f_u:.3f}')